# Bagian B paralel — eksperimen permintaan reviewer (ablasi, pembanding, multi-seed)

Notebook ini menjalankan **Bagian B** dari `fundus_effb0_se_sesuai_artikel.ipynb` secara terpisah dan lebih cepat. Definisi model, augmentasi, jadwal LR, pembagian 70:15:15, dan metrik disalin dari notebook utama. **Satu perubahan (15 Nov 2026):** early stopping dan pemilihan bobot terbaik memantau `val_accuracy`, bukan `val_loss` (lihat `PANTAU` di sel Konfigurasi). Alasannya: pada 30 run pertama `val_loss` naik turun tajam sehingga sebagian run terpotong dini (mis. CBAM seed 3 berhenti di epoch 43, terbaik 13, akurasi uji 74,6%), dan akurasi uji berkorelasi 0,78 dengan epoch terbaik. Perbandingan antarvarian jadi lebih ditentukan oleh kapan run berhenti daripada oleh arsitekturnya.

**Yang membuatnya lebih cepat**

| Cara | Efek | Mengubah hasil? |
|---|---|---|
| Dua run berjalan bersamaan, satu per GPU (T4 x2) | ± 2x | Tidak |
| Bagian A tidak dijalankan ulang | hemat ± 1 jam per sesi | Tidak |
| Urutan seed-dulu + `BATAS_JAM` + lanjut otomatis dari CSV | sesi tidak terbuang | Tidak |

**Cara pakai di Kaggle**: Accelerator **GPU T4 x2**, Internet ON, tambahkan dataset *eye-diseases-classification*. Jalankan sekali dengan `UJI_CEPAT = True` (beberapa menit) untuk memastikan alurnya jalan, lalu set `False` dan pakai **Save Version → Save & Run All (Commit)** agar berjalan di latar belakang.


### B0. Keras 2 dan deteksi GPU (jalankan pertama)

In [ ]:
import os, sys, subprocess, importlib.util
from importlib.metadata import version, PackageNotFoundError
try:
    _tf = tuple(int(v) for v in version('tensorflow').split('.')[:2])
except PackageNotFoundError:
    _tf = (0, 0)
if _tf >= (2, 16):                                # TF >= 2.16 bawaannya Keras 3 -> pakai tf-keras (Keras 2)
    os.environ['TF_USE_LEGACY_KERAS'] = '1'       # diwarisi juga oleh proses pekerja
    if importlib.util.find_spec('tf_keras') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', f'tf-keras~={_tf[0]}.{_tf[1]}.0'], check=True)
    import tf_keras; print('Keras 2 aktif (tf_keras', tf_keras.__version__ + ')')
else:
    print('TensorFlow', '.'.join(map(str, _tf)), '-> sudah Keras 2')

# Hitung GPU, lalu SEMBUNYIKAN dari proses notebook ini: GPU hanya dipakai oleh proses pekerja (satu pekerja per GPU).
try:
    N_GPU = len([l for l in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.splitlines() if l.startswith('GPU ')])
except Exception:
    N_GPU = 0
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
print('GPU terdeteksi:', N_GPU)


### Konfigurasi

In [ ]:
import json, time, glob, math, itertools
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
_CANDS = ['/kaggle/input/eye-diseases-classification/dataset',
          '/kaggle/input/datasets/gunavenkatdoddi/eye-diseases-classification/dataset']
DATASET_DIR = next((d for d in _CANDS if os.path.isdir(d)), None)
if DATASET_DIR is None:
    import kagglehub
    DATASET_DIR = os.path.join(kagglehub.dataset_download('gunavenkatdoddi/eye-diseases-classification'), 'dataset')
OUT_DIR = '/kaggle/working/hasil' if os.path.isdir('/kaggle/working') else 'hasil'

# ---- konfigurasi pelatihan: HARUS sama dengan Bagian A di notebook utama
SEED, IMG_SIZE, BATCH_SIZE, EPOCHS, VAL_SPLIT, SE_RATIO = 123, (224, 224), 32, 150, 0.10, 8
USE_STOP_CRITERION    = True     # tidak dipakai di Bagian B (run_experiment memanggil use_stop=False)
BN_BACKBONE_INFERENCE = True
LR_PER_ITERASI        = False
PANTAU                = 'val_accuracy'   # besaran untuk early stopping + bobot terbaik. 'val_loss' = resep lama (Bagian A)
PATIENCE              = 30

# ---- apa yang dijalankan. Urutan: semua varian untuk seed pertama dulu, lalu seed berikutnya,
#      sehingga kalau waktu habis tabel tetap lengkap (hanya jumlah seed-nya lebih sedikit).
RUN_VARIANTS = [('efficientnetb0', 'se', 8), ('efficientnetb0', 'none', 8), ('efficientnetb0', 'se', 4),
                ('efficientnetb0', 'se', 16), ('efficientnetb0', 'eca', 8), ('efficientnetb0', 'cbam', 8),
                ('resnet50', 'none', 8), ('mobilenetv2', 'none', 8)]
RUN_SEEDS    = [0, 1, 2, 3, 4]  # 40 run; perlu lebih dari satu sesi Kaggle (hasil sesi sebelumnya ditambahkan sebagai Input)

# ---- percepatan
N_PARALEL       = None    # None = satu pekerja per GPU (T4 x2 -> 2 run sekaligus). Isi 1 untuk berurutan.
BATAS_JAM       = 11.2    # semua run harus SELESAI sebelum jam ini (batas sesi Kaggle 12 jam). Run baru hanya dimulai bila perkiraan terlamanya masih muat.
MAKS_MENIT      = {'efficientnetb0': 63, 'resnet50': 97, 'mobilenetv2': 40}   # lama terpanjang satu run 150 epoch di T4 (terukur 14 Nov, +10%)
UJI_CEPAT       = False   # True: uji alur beberapa menit (2 run, 2 epoch, sedikit citra). Jalankan sekali dulu sebelum run panjang.

os.makedirs(OUT_DIR, exist_ok=True)
T_MULAI = time.time()
print('DATASET_DIR :', DATASET_DIR); print('OUT_DIR     :', os.path.abspath(OUT_DIR))


### B1. Data, duplikat, dan pembagian 70:15:15 (identik dengan notebook utama)

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
import tensorflow as tf
keras = tf.keras
from sklearn.model_selection import train_test_split

# --- urutan berkas identik dengan notebook utama (seed 123)
common = dict(validation_split=VAL_SPLIT, seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
              shuffle=True, label_mode='int')
train_raw = keras.utils.image_dataset_from_directory(DATASET_DIR, subset='training', **common)
val_raw   = keras.utils.image_dataset_from_directory(DATASET_DIR, subset='validation', **common)
class_names = train_raw.class_names
NUM_CLASSES = len(class_names)
CLS_ID = {c: i for i, c in enumerate(class_names)}
GLAUCOMA = CLS_ID['glaucoma']; NORMAL = CLS_ID['normal']

train_files = list(train_raw.file_paths); val_files = list(val_raw.file_paths)
assert not set(train_files) & set(val_files), 'ada berkas yang muncul di latih dan validasi!'
lbl = lambda p: os.path.basename(os.path.dirname(p))
split_df = pd.DataFrame({'file': train_files + val_files,
                         'kelas': [lbl(p) for p in train_files + val_files],
                         'subset': ['latih'] * len(train_files) + ['validasi'] * len(val_files)})
print(pd.crosstab(split_df.kelas, split_df.subset, margins=True, margins_name='Total'))

# --- deteksi duplikat pHash (sama dengan A3)
from scipy.fftpack import dct
def phash(path, hash_size=8, highfreq=4):
    img = Image.open(path).convert('L').resize((hash_size * highfreq,) * 2, Image.LANCZOS)
    a = np.asarray(img, dtype=np.float32)
    d = dct(dct(a, axis=0, norm='ortho'), axis=1, norm='ortho')[:hash_size, :hash_size]
    bits = (d > np.median(d)).flatten()
    return np.packbits(bits).view('>u8')[0]

rows = []
for p in split_df.file:
    with Image.open(p) as im:
        w, h = im.size
    rows.append((w, h, os.path.splitext(p)[1].lower(), phash(p)))
_meta = pd.DataFrame(rows, index=split_df.index, columns=['lebar', 'tinggi', 'ekstensi', 'phash'])
for _c in _meta.columns: split_df[_c] = _meta[_c]

H = split_df.phash.values.astype(np.uint64)
def hamming(a, B):
    x = np.bitwise_xor(B, a)
    return np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(1)
THR = 4   # jarak Hamming <= 4 dari 64 bit dianggap duplikat/hampir duplikat
pairs = []
for i in range(len(H)):
    d = hamming(H[i], H[i + 1:])
    for j in np.nonzero(d <= THR)[0]:
        pairs.append((i, i + 1 + j, int(d[j])))
dup = pd.DataFrame(pairs, columns=['i', 'j', 'hamming'])
if len(dup):
    dup['kelas_i'] = split_df.kelas.values[dup.i]; dup['kelas_j'] = split_df.kelas.values[dup.j]
    dup['subset_i'] = split_df.subset.values[dup.i]; dup['subset_j'] = split_df.subset.values[dup.j]
    dup['file_i'] = split_df.file.values[dup.i]; dup['file_j'] = split_df.file.values[dup.j]
print('Pasangan (hampir) duplikat:', len(dup))

# --- kelompok duplikat (union-find) dan pembagian 70:15:15 terstratifikasi (sama dengan B1)
parent = list(range(len(split_df)))
def find(a):
    while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
    return a
for i, j in zip(dup.i if len(dup) else [], dup.j if len(dup) else []): parent[find(i)] = find(j)
split_df['grup'] = [find(i) for i in range(len(split_df))]
konflik = split_df.groupby('grup').kelas.nunique(); konflik = konflik[konflik > 1].index
print('Grup duplikat dengan label berbeda (dibuang dari Bagian B):', len(konflik))
dfB = split_df[~split_df.grup.isin(konflik)].copy()
grp = dfB.groupby('grup').agg(kelas=('kelas', 'first')).reset_index()
g_tr, g_tmp = train_test_split(grp, test_size=0.30, stratify=grp.kelas, random_state=SEED)
g_va, g_te = train_test_split(g_tmp, test_size=0.50, stratify=g_tmp.kelas, random_state=SEED)
dfB['subset'] = dfB.grup.map({**{g: 'latih' for g in g_tr.grup}, **{g: 'validasi' for g in g_va.grup}, **{g: 'uji' for g in g_te.grup}})
dfB.to_csv(f'{OUT_DIR}/pembagian_70_15_15.csv', index=False)
print(pd.crosstab(dfB.kelas, dfB.subset, margins=True))
P = {s: dfB.file[dfB.subset == s].tolist() for s in ['latih', 'validasi', 'uji']}
if UJI_CEPAT:
    P = {s: v[::max(1, len(v) // n)][:n] for (s, v), n in zip(P.items(), [96, 48, 48])}
    print('UJI_CEPAT:', {s: len(v) for s, v in P.items()})


### Pustaka pekerja (ditulis ke `fundus_lib.py`)

In [ ]:
%%writefile fundus_lib.py
# Pekerja Bagian B: melatih SATU konfigurasi pada SATU GPU. Dipanggil oleh sel "B2" sebagai proses terpisah:
#   python fundus_lib.py <state.json> <job.json> <hasil.json>
# Definisi model, augmentasi, callback, dan metrik disalin apa adanya dari notebook utama.
import os, sys, json, time, math
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
import tensorflow as tf
keras = tf.keras
for _g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(_g, True)
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

ST = json.load(open(sys.argv[1], encoding='utf-8'))
globals().update(ST['konfig'])            # SEED, IMG_SIZE, BATCH_SIZE, EPOCHS, SE_RATIO, USE_STOP_CRITERION, BN_BACKBONE_INFERENCE, LR_PER_ITERASI, OUT_DIR, MIXED_PRECISION, PANTAU, PATIENCE
IMG_SIZE = tuple(IMG_SIZE)
class_names = ST['class_names']; NUM_CLASSES = len(class_names); CLS_ID = {c: i for i, c in enumerate(class_names)}
P = ST['P']
lbl = lambda p: os.path.basename(os.path.dirname(p))
AUTOTUNE = tf.data.AUTOTUNE
NOISE_STD = 0.05
L = keras.layers
if MIXED_PRECISION:
    keras.mixed_precision.set_global_policy('mixed_float16')

def load_img(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, method='bilinear')     # sama dengan image_dataset_from_directory
    img.set_shape(IMG_SIZE + (3,))
    return img

def ds_from_paths(paths, training=False, seed=SEED):
    labels = [CLS_ID[lbl(p)] for p in paths]
    ds = tf.data.Dataset.from_tensor_slices((list(paths), labels))
    ds = ds.map(lambda p, y: (load_img(p), tf.one_hot(y, NUM_CLASSES)), num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(1000, seed=seed, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

@keras.utils.register_keras_serializable(package='fundus')   # keras.utils ada di Keras 2 dan 3 (keras.saving tidak dimuat lewat tf.keras di Keras 3)
class RandomCropResize(keras.layers.Layer):
    """Potong acak crop×crop lalu ubah kembali ke ukuran semula; HANYA saat pelatihan."""
    def __init__(self, crop=200, out=224, seed=None, **kw):
        super().__init__(**kw); self.crop, self.out, self.seed = crop, out, seed
    def call(self, x, training=None):
        if not training:
            return x
        b = tf.shape(x)[0]
        x = tf.image.random_crop(x, tf.stack([b, self.crop, self.crop, 3]), seed=self.seed)
        return tf.image.resize(x, (self.out, self.out))
    def get_config(self):
        return {**super().get_config(), 'crop': self.crop, 'out': self.out, 'seed': self.seed}

def make_augmentation(seed=SEED):
    return keras.Sequential([
        RandomCropResize(200, 224, seed=seed),
        keras.layers.RandomFlip('horizontal', seed=seed),
        keras.layers.RandomRotation(0.1, seed=seed),          # faktor 0,1 = ±36°
        keras.layers.RandomZoom(0.1, seed=seed),
        keras.layers.RandomContrast(0.1, seed=seed),
        keras.layers.RandomBrightness(0.1, seed=seed),
        keras.layers.RandomTranslation(0.1, 0.1, seed=seed),
        keras.layers.GaussianNoise(NOISE_STD, seed=seed),
    ], name='augmentasi')

def attention_block(x, kind='se', ratio=8):
    C = x.shape[-1]
    if kind == 'none':
        return x
    if kind == 'se':
        s = L.GlobalAveragePooling2D(name='se_squeeze')(x)
        s = L.Reshape((1, 1, C), name='se_reshape')(s)
        s = L.Dense(C // ratio, activation='relu', name='se_fc1')(s)
        s = L.Dense(C, activation='sigmoid', name='se_fc2')(s)
        return L.Multiply(name='se_scale')([x, s])
    if kind == 'eca':           # Wang dkk. (ECA-Net), k adaptif
        k = int(abs((math.log2(C) + 1) / 2)); k = k if k % 2 else k + 1
        s = L.GlobalAveragePooling2D(name='eca_gap')(x)
        s = L.Reshape((C, 1), name='eca_reshape1')(s)
        s = L.Conv1D(1, k, padding='same', use_bias=False, name='eca_conv')(s)
        s = L.Activation('sigmoid', name='eca_sigmoid')(s)
        s = L.Reshape((1, 1, C), name='eca_reshape2')(s)
        return L.Multiply(name='eca_scale')([x, s])
    if kind == 'cbam':          # Woo dkk. (CBAM): atensi kanal lalu spasial
        fc1 = L.Dense(C // ratio, activation='relu', name='cbam_fc1'); fc2 = L.Dense(C, name='cbam_fc2')
        a = fc2(fc1(L.GlobalAveragePooling2D(name='cbam_gap')(x)))
        m = fc2(fc1(L.GlobalMaxPooling2D(name='cbam_gmp')(x)))
        ca = L.Activation('sigmoid', name='cbam_ch_sig')(L.Add(name='cbam_ch_add')([a, m]))
        x = L.Multiply(name='cbam_ch_scale')([x, L.Reshape((1, 1, C), name='cbam_ch_reshape')(ca)])
        avg = L.Lambda(lambda t: tf.reduce_mean(t, -1, keepdims=True), name='cbam_sp_mean')(x)
        mx = L.Lambda(lambda t: tf.reduce_max(t, -1, keepdims=True), name='cbam_sp_max')(x)
        sa = L.Conv2D(1, 7, padding='same', activation='sigmoid', name='cbam_sp_conv')(L.Concatenate(name='cbam_sp_cat')([avg, mx]))
        return L.Multiply(name='cbam_sp_scale')([x, sa])
    raise ValueError(kind)

def make_backbone(name):
    if name == 'efficientnetb0':
        return keras.applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)), None
    if name == 'resnet50':
        return keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)), keras.applications.resnet50.preprocess_input
    if name == 'mobilenetv2':
        return keras.applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)), keras.applications.mobilenet_v2.preprocess_input
    raise ValueError(name)

def build_model(backbone='efficientnetb0', attention='se', ratio=SE_RATIO, seed=SEED):
    base, prep = make_backbone(backbone)
    base.trainable = True                      # seluruh lapisan di-fine-tune
    inputs = L.Input(IMG_SIZE + (3,), name='citra')
    x = make_augmentation(seed)(inputs)        # PERBAIKAN #1: keluaran augmentasi dipakai
    if prep is not None:                        # EfficientNet Keras sudah memuat rescaling+normalisasi
        x = L.Lambda(prep, name='preprocess')(x)
    # PERBAIKAN #2: training tidak dikunci ke True. PERBAIKAN #9: BN backbone dalam mode inferensi (bobot tetap dilatih)
    x = base(x, training=False) if BN_BACKBONE_INFERENCE else base(x)
    x = attention_block(x, attention, ratio)
    x = L.GlobalAveragePooling2D(name='head_gap')(x)
    x = L.BatchNormalization(name='head_bn1')(x)
    x = L.Dense(128, activation='relu', name='head_fc')(x)
    x = L.BatchNormalization(name='head_bn2')(x)
    x = L.Dropout(0.3, name='head_dropout')(x)
    out = L.Dense(NUM_CLASSES, activation='softmax', name='keluaran', dtype='float32')(x)   # float32 agar aman bila mixed precision
    return keras.Model(inputs, out, name=f'{backbone}_{attention}{ratio if attention in ("se","cbam") else ""}'), base

def cyclic_lr(epoch, lr=None, base_lr=1e-5, max_lr=1e-3, step_size=2000):
    cycle = np.floor(1 + epoch / (2 * step_size))
    x = np.abs(epoch / step_size - 2 * cycle + 1)
    return float(base_lr + (max_lr - base_lr) * max(0, 1 - x))

class CyclicLRIter(keras.callbacks.Callback):
    """Persamaan (4) dan (5) dengan t = nomor iterasi (batch), bukan nomor epoch."""
    def on_train_begin(self, logs=None): self.it = 0
    def on_train_batch_begin(self, batch, logs=None):
        keras.backend.set_value(self.model.optimizer.learning_rate, cyclic_lr(self.it)); self.it += 1
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None: logs['lr'] = float(keras.backend.get_value(self.model.optimizer.learning_rate))

class StopCriterion(keras.callbacks.Callback):
    def __init__(self): super().__init__(); self.fired_epoch = None
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if logs.get('val_accuracy', 0) >= 0.95 and logs.get('val_loss', 9) < 0.13:
            self.fired_epoch = epoch + 1; self.model.stop_training = True

class Timer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.t0 = time.time()
    def on_train_end(self, logs=None): self.seconds = time.time() - self.t0

class BestKeeper(keras.callbacks.Callback):
    """Simpan bobot epoch terbaik di memori dan pasang kembali di akhir pelatihan, juga bila early stopping TIDAK terpicu
    (EarlyStopping(restore_best_weights) tidak menjamin itu saat epoch habis)."""
    def __init__(self, monitor):
        super().__init__(); self.monitor = monitor; self.sign = -1 if 'loss' in monitor else 1
        self.best = -np.inf; self.best_epoch = 0; self.w = None
    def on_epoch_end(self, epoch, logs=None):
        v = self.sign * float((logs or {}).get(self.monitor, -np.inf * self.sign))
        if v > self.best: self.best, self.best_epoch, self.w = v, epoch + 1, self.model.get_weights()
    def on_train_end(self, logs=None):
        if self.w is not None: self.model.set_weights(self.w)

def make_callbacks(ckpt_path, use_stop=USE_STOP_CRITERION, log_path=None):
    stop, timer = StopCriterion(), Timer()
    es = keras.callbacks.EarlyStopping(monitor=PANTAU, mode=('min' if 'loss' in PANTAU else 'max'), patience=PATIENCE)
    es.keeper = BestKeeper(PANTAU)
    cbs = [CyclicLRIter() if LR_PER_ITERASI else keras.callbacks.LearningRateScheduler(cyclic_lr), es, es.keeper, timer]
    if use_stop: cbs.insert(0, stop)
    if log_path: cbs.append(keras.callbacks.CSVLogger(log_path))
    return cbs, stop, es, timer

def predict_paths(m, paths):
    ds = ds_from_paths(paths)
    probs = m.predict(ds, verbose=0)
    y = np.array([CLS_ID[lbl(p)] for p in paths])
    return y, probs

def confusion(y, yp, k):
    cm = np.zeros((k, k), int); np.add.at(cm, (y, yp), 1); return cm

def per_class(cm):
    n = cm.sum(); out = []
    for i in range(len(cm)):
        tp = cm[i, i]; fn = cm[i].sum() - tp; fp = cm[:, i].sum() - tp; tn = n - tp - fn - fp
        prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0
        out.append(dict(tp=tp, fp=fp, fn=fn, tn=tn, presisi=prec, sensitivitas=rec,
                        spesifisitas=tn / (tn + fp) if tn + fp else 0.0,
                        f1=2 * prec * rec / (prec + rec) if prec + rec else 0.0, support=tp + fn))
    return out

def kappa(cm):
    n = cm.sum(); po = np.trace(cm) / n; pe = (cm.sum(0) * cm.sum(1)).sum() / n**2
    return (po - pe) / (1 - pe)

def summary_metrics(y, yp, k):
    cm = confusion(y, yp, k); pc = per_class(cm); sup = np.array([d['support'] for d in pc])
    macro = {m: np.mean([d[m] for d in pc]) for m in ['presisi', 'sensitivitas', 'spesifisitas', 'f1']}
    weighted = {m: np.average([d[m] for d in pc], weights=sup) for m in ['presisi', 'sensitivitas', 'spesifisitas', 'f1']}
    return dict(cm=cm, per_class=pc, macro=macro, weighted=weighted, akurasi=np.trace(cm) / cm.sum(), kappa=kappa(cm))

_make_aug = make_augmentation
def make_augmentation(seed=SEED):
    # augmentasi selalu float32 (RandomContrast tidak punya kernel float16); hanya backbone dan head yang float16
    if not MIXED_PRECISION: return _make_aug(seed)
    keras.mixed_precision.set_global_policy('float32'); aug = _make_aug(seed)
    keras.mixed_precision.set_global_policy('mixed_float16'); return aug

class EpochLog(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None): self.t = time.time()
    def on_epoch_end(self, epoch, logs=None):
        l = logs or {}
        print(f"epoch {epoch+1}/{EPOCHS} loss {l.get('loss', 0):.4f} acc {l.get('accuracy', 0):.4f} "
              f"val_loss {l.get('val_loss', 0):.4f} val_acc {l.get('val_accuracy', 0):.4f} ({time.time()-self.t:.0f}s)", flush=True)

def run_experiment(backbone, attention, ratio, seed):
    keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    m, _ = build_model(backbone, attention, ratio, seed=seed)
    m.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0), loss='categorical_crossentropy', metrics=['accuracy'])
    ytr = np.array([CLS_ID[lbl(p)] for p in P['latih']]); cnt = np.bincount(ytr, minlength=NUM_CLASSES)
    cw = {i: len(ytr) / (NUM_CLASSES * c) for i, c in enumerate(cnt)}
    ck = f'{OUT_DIR}/tmp_{backbone}_{attention}{ratio}_s{seed}.keras'
    cbs, _, es, timer = make_callbacks(ck, use_stop=False)
    h = m.fit(ds_from_paths(P['latih'], training=True, seed=seed), validation_data=ds_from_paths(P['validasi']),
              epochs=EPOCHS, class_weight=cw, callbacks=cbs + [EpochLog()], verbose=0)
    yt, pr = predict_paths(m, P['uji']); pr = pr.astype('float32'); s = summary_metrics(yt, pr.argmax(1), NUM_CLASSES)
    ybt = label_binarize(yt, classes=range(NUM_CLASSES))
    out = dict(backbone=backbone, atensi=attention, rasio=ratio, seed=seed, param=m.count_params(),
               epoch=len(h.history['loss']), epoch_terbaik=int(es.keeper.best_epoch), val_akurasi_terbaik=float(max(h.history['val_accuracy'])), menit=timer.seconds / 60,
               akurasi=s['akurasi'], macro_f1=s['macro']['f1'], kappa=s['kappa'], auc_micro=roc_auc_score(ybt, pr, average='micro'),
               **{f'sens_{c}': d['sensitivitas'] for c, d in zip(class_names, s['per_class'])})
    np.save(f'{OUT_DIR}/prob_uji_{backbone}_{attention}{ratio}_s{seed}.npy', pr)
    if os.path.exists(ck): os.remove(ck)
    return out

if __name__ == '__main__':
    job = json.load(open(sys.argv[2], encoding='utf-8'))
    print('GPU terlihat:', [d.name for d in tf.config.list_physical_devices('GPU')] or 'CPU saja', '| job:', job, flush=True)
    row = run_experiment(job['backbone'], job['atensi'], job['rasio'], job['seed'])
    json.dump(row, open(sys.argv[3], 'w', encoding='utf-8'), default=float)
    print('SELESAI', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()}, flush=True)


### B2. Jalankan antrean: satu proses pekerja per GPU

Log tiap run ada di `hasil/log_B/<nama>.log`. Baris `...` adalah status berkala (epoch terakhir tiap pekerja).

In [ ]:
KONFIG = dict(SEED=SEED, IMG_SIZE=list(IMG_SIZE), BATCH_SIZE=BATCH_SIZE, EPOCHS=(2 if UJI_CEPAT else EPOCHS), SE_RATIO=SE_RATIO,
              USE_STOP_CRITERION=USE_STOP_CRITERION, BN_BACKBONE_INFERENCE=BN_BACKBONE_INFERENCE, LR_PER_ITERASI=LR_PER_ITERASI,
              OUT_DIR=OUT_DIR, MIXED_PRECISION=False, PANTAU=PANTAU, PATIENCE=PATIENCE)   # mixed precision diuji: loss NaN pada EfficientNet -> tidak dipakai
TAG = f'bn{int(BN_BACKBONE_INFERENCE)}_lriter{int(LR_PER_ITERASI)}_ep{KONFIG["EPOCHS"]}_{PANTAU}_p{PATIENCE}'   # run dengan konfigurasi lain tidak dianggap selesai
B_CSV = f'{OUT_DIR}/' + ('eksperimen_B_ujicepat.csv' if UJI_CEPAT else 'eksperimen_B.csv')
LOG_DIR = f'{OUT_DIR}/log_B'; os.makedirs(LOG_DIR, exist_ok=True)
STATE = f'{OUT_DIR}/state_B.json'
json.dump(dict(konfig=KONFIG, class_names=class_names, P=P), open(STATE, 'w', encoding='utf-8'))

# --- lanjutkan dari sesi sebelumnya: CSV lokal + CSV dari keluaran notebook lama yang ditambahkan sebagai Input
KEY = ['backbone', 'atensi', 'rasio', 'seed', 'konfig']
lama = [B_CSV] + ([] if UJI_CEPAT else glob.glob('/kaggle/input/**/eksperimen_B.csv', recursive=True))
done = [pd.read_csv(f) for f in lama if os.path.exists(f)]
done = pd.concat(done, ignore_index=True) if done else pd.DataFrame(columns=KEY)
if 'konfig' not in done.columns: done['konfig'] = 'lama'
done['konfig'] = done['konfig'].fillna('lama')
done = done.drop_duplicates(KEY, keep='last').reset_index(drop=True)
sudah = set((a, b, int(c), int(d)) for a, b, c, d in done[done.konfig == TAG][KEY[:4]].values.tolist())
jobs = [dict(backbone=bb, atensi=att, rasio=r, seed=sd) for sd, (bb, att, r) in itertools.product(RUN_SEEDS, RUN_VARIANTS)]
jobs = [j for j in jobs if (j['backbone'], j['atensi'], j['rasio'], j['seed']) not in sudah]
if UJI_CEPAT: jobs = jobs[:2]
print(f'Sudah selesai (konfigurasi {TAG}): {len(sudah)} run | antrean: {len(jobs)} run')

# --- unduh bobot ImageNet lebih dulu (di CPU) agar dua pekerja tidak mengunduh berkas yang sama bersamaan
_apps = {'efficientnetb0': keras.applications.EfficientNetB0, 'resnet50': keras.applications.ResNet50, 'mobilenetv2': keras.applications.MobileNetV2}
for bb in sorted({j['backbone'] for j in jobs}):
    _apps[bb](weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)); keras.backend.clear_session()

slots = list(range(N_GPU)) if N_GPU else [None]            # None = CPU
if N_PARALEL: slots = slots[:N_PARALEL]
print(f'Pekerja paralel: {len(slots)} ->', ['GPU %d' % s if s is not None else 'CPU' for s in slots])

nama_job = lambda j: f"{j['backbone']}_{j['atensi']}{j['rasio']}_s{j['seed']}"
def ekor(path, n=1):
    try:
        with open(path, encoding='utf-8', errors='replace') as f: return [l.rstrip() for l in f.readlines() if l.strip()][-n:]
    except OSError: return []
menit = lambda: (time.time() - T_MULAI) / 60

aktif, gagal, antre, t_status = {}, [], list(jobs), time.time()
while antre or aktif:
    for s in slots:                                         # isi slot kosong
        muat = [k for k, q in enumerate(antre) if menit() + MAKS_MENIT.get(q['backbone'], 100) * KONFIG['EPOCHS'] / 150 <= BATAS_JAM * 60]
        if s not in aktif and muat:                         # ambil run pertama di antrean yang masih sempat selesai
            j = antre.pop(muat[0]); nm = nama_job(j)
            jf, rf, lf = f'{LOG_DIR}/{nm}.job.json', f'{LOG_DIR}/{nm}.hasil.json', f'{LOG_DIR}/{nm}.log'
            json.dump(j, open(jf, 'w'))
            if os.path.exists(rf): os.remove(rf)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=('-1' if s is None else str(s)), PYTHONIOENCODING='utf-8')
            p = subprocess.Popen([sys.executable, 'fundus_lib.py', STATE, jf, rf], stdout=open(lf, 'w', encoding='utf-8'), stderr=subprocess.STDOUT, env=env)
            aktif[s] = (p, j, rf, lf)
            print(f'[{menit():6.1f} mnt] MULAI   {nm} di ' + ('CPU' if s is None else f'GPU {s}'), flush=True)
    if not aktif: break                                     # antrean masih ada tetapi batas waktu tercapai
    time.sleep(5 if UJI_CEPAT else 20)
    for s, (p, j, rf, lf) in list(aktif.items()):           # ambil yang sudah selesai
        if p.poll() is None: continue
        del aktif[s]; nm = nama_job(j)
        if p.returncode == 0 and os.path.exists(rf):
            row = json.load(open(rf)); row['konfig'] = TAG
            done = pd.concat([done, pd.DataFrame([row])], ignore_index=True).drop_duplicates(KEY, keep='last'); done.to_csv(B_CSV, index=False)
            print(f"[{menit():6.1f} mnt] SELESAI {nm}: akurasi uji {row['akurasi']:.4f}, macro-F1 {row['macro_f1']:.4f}, "
                  f"epoch {row['epoch']} (terbaik {row['epoch_terbaik']}), {row['menit']:.1f} mnt", flush=True)
            print('    BARIS_CSV ' + json.dumps(row), flush=True)   # cadangan: CSV bisa disusun ulang dari teks keluaran ini
        else:
            gagal.append(nm); print(f'!! GAGAL {nm} (kode {p.returncode}). Akhir log:\n   ' + '\n   '.join(ekor(lf, 25)), flush=True)
    if time.time() - t_status > (20 if UJI_CEPAT else 600):  # status berkala
        t_status = time.time()
        for s, (p, j, rf, lf) in aktif.items():
            print(f"[{menit():6.1f} mnt]   ...     {nama_job(j)}: {(ekor(lf) or ['memuat'])[0][:110]}", flush=True)

print(f'\nSelesai sesi ini. Run tersimpan untuk konfigurasi {TAG}: {int((done.konfig == TAG).sum())} | belum dijalankan: {len(antre)} | gagal: {gagal}')
if antre: print('Untuk melanjutkan: simpan versi ini, tambahkan keluarannya sebagai Input di sesi berikutnya, lalu jalankan lagi. Run yang selesai dilewati otomatis.')


### B3. Tabel ablasi dan pembanding (rata-rata ± SD pada data uji)

In [ ]:
fmt = lambda x, d=2: f'{x:.{d}f}'.replace('.', ',')
done = pd.read_csv(B_CSV) if os.path.exists(B_CSV) else pd.DataFrame(columns=KEY); done = done[done.konfig == TAG]
assert len(done), 'Belum ada run yang selesai untuk konfigurasi ini; periksa log di hasil/log_B'
done['model'] = done.backbone + ' + ' + done.atensi.where(done.atensi == 'none', done.atensi + ' r=' + done.rasio.astype(str))
metr = ['akurasi', 'macro_f1', 'kappa', 'auc_micro', 'sens_glaucoma']
agg = done.groupby('model').agg(n_seed=('seed', 'nunique'), param=('param', 'first'),
                                **{f'{m}_mean': (m, 'mean') for m in metr}, **{f'{m}_sd': (m, 'std') for m in metr})
tabel = pd.DataFrame({'Model': agg.index, 'n seed': agg.n_seed.values, 'Parameter': agg.param.map('{:,}'.format).str.replace(',', '.').values})
for m in metr:
    k = 1 if m in ('kappa', 'auc_micro') else 100; d = 3 if k == 1 else 2
    tabel[m] = [f'{fmt(k*a, d)} ± {fmt(k*b, d)}' for a, b in zip(agg[f'{m}_mean'], agg[f'{m}_sd'].fillna(0))]
display(tabel); tabel.to_csv(f'{OUT_DIR}/tabel_ablasi_pembanding.csv', index=False)

# selisih berpasangan per seed: SE r=8 vs tanpa SE
a = done[(done.backbone == 'efficientnetb0') & (done.atensi == 'se') & (done.rasio == 8)].set_index('seed')
b = done[(done.backbone == 'efficientnetb0') & (done.atensi == 'none')].set_index('seed')
common_seeds = a.index.intersection(b.index)
if len(common_seeds) >= 2:
    diff = (a.loc[common_seeds, 'akurasi'] - b.loc[common_seeds, 'akurasi']) * 100
    from scipy import stats
    t, p = stats.ttest_rel(a.loc[common_seeds, 'akurasi'], b.loc[common_seeds, 'akurasi'])
    print(f'Selisih akurasi SE r=8 − tanpa SE: {diff.mean():.2f} ± {diff.std():.2f} poin persentase (n={len(common_seeds)} seed, uji-t berpasangan p = {p:.3f})')